In [ ]:
# CAPSTONE PROJECT 1
# NHANES Dataset Analysis Using NumPy, Matplotlib, and Statistics

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Step 1: Load data and remove rows with missing (nan) values
male = np.genfromtxt('nhanes_adult_male_bmx_2020.csv', delimiter=',', skip_header=1)
female = np.genfromtxt('nhanes_adult_female_bmx_2020.csv', delimiter=',', skip_header=1)

# Remove rows with any missing values
male = male[~np.isnan(male).any(axis=1)]
female = female[~np.isnan(female).any(axis=1)]

# Step 2: Histograms for female (top) and male (bottom) weights
plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.hist(female[:, 0], bins=30, color='pink', edgecolor='black')
plt.title("Female Weight Distribution")

plt.subplot(2, 1, 2)
plt.hist(male[:, 0], bins=30, color='blue', edgecolor='black')
plt.title("Male Weight Distribution")

plt.xlim(30, 150)
plt.tight_layout()
plt.show()

# Step 3: Boxplot comparing male and female weights
plt.boxplot([female[:, 0], male[:, 0]], tick_labels=['Female', 'Male'])
plt.title("Boxplot of Weights")
plt.ylabel("Weight (kg)")
plt.grid()
plt.show()

# Step 4: Descriptive statistics for weights
def describe(data):
    return {
        'Mean': round(np.mean(data), 2),
        'Median': round(np.median(data), 2),
        'Std Dev': round(np.std(data), 2),
        'Min': round(np.min(data), 2),
        'Max': round(np.max(data), 2),
        'Skew (approx)': round(3 * (np.mean(data) - np.median(data)) / np.std(data), 2)
    }

print("Female Weight Stats:", describe(female[:, 0]))
print("Male Weight Stats:", describe(male[:, 0]))

# Step 5: Add BMI as the 8th column in female matrix
height_m = female[:, 1] / 100
bmi = female[:, 0] / (height_m ** 2)
female = np.column_stack((female, bmi))

# Step 6: Add Waist/Height and Waist/Hip ratios to female and male matrices
female_waist_height = female[:, 6] / female[:, 1]
female_waist_hip = female[:, 6] / female[:, 5]
female = np.column_stack((female, female_waist_height, female_waist_hip))

male_waist_height = male[:, 6] / male[:, 1]
male_waist_hip = male[:, 6] / male[:, 5]
male = np.column_stack((male, male_waist_height, male_waist_hip))

# Step 7: Standardize (z-score) the female dataset
mean = np.mean(female, axis=0)
std = np.std(female, axis=0)
zfemale = (female - mean) / std

# Step 8: Pairplot and correlation
selected_cols = [0, 1, 6, 5, 7]  # weight, height, waist, hip, BMI
df = pd.DataFrame(zfemale[:, selected_cols], columns=['Weight', 'Height', 'Waist', 'Hip', 'BMI'])
sns.pairplot(df)
plt.suptitle("Standardized Female Measurements - Pairplot", y=1.02)
plt.show()

print("Pearson Correlation:\n", df.corr(method='pearson').round(2))
print("\nSpearman Correlation:\n", df.corr(method='spearman').round(2))

# Step 9: Boxplot for waist ratios (only if shape is valid)
if female.shape[1] >= 10 and male.shape[1] >= 9:
    plt.boxplot([
        female[:, 8],
        male[:, 7],
        female[:, 9],
        male[:, 8]
    ], tick_labels=['F Waist/Height', 'M Waist/Height', 'F Waist/Hip', 'M Waist/Hip'])
    plt.title("Comparison of Waist Ratios")
    plt.ylabel("Ratio")
    plt.grid()
    plt.show()
else:
    print("Error: Ratio columns not added properly. Please check earlier steps.")

# Step 10: Print pros and cons of each metric
print("""
BMI:
+ Easy to calculate
+ Useful for population-level screening
- Does not differentiate fat vs. muscle

Waist-to-Height Ratio:
+ Better indicator of central obesity
+ Stronger link to cardiovascular risk

Waist-to-Hip Ratio:
+ Reflects fat distribution pattern
- May be less predictive of disease risk than waist/height
""")

# Step 11: Print standardized values of 5 lowest & highest BMI females
bmi_index = np.argsort(female[:, 7])
lowest5 = bmi_index[:5]
highest5 = bmi_index[-5:]

print("\nLowest 5 BMI (Z-scores):\n", zfemale[lowest5][:, selected_cols])
print("\nHighest 5 BMI (Z-scores):\n", zfemale[highest5][:, selected_cols])
